In [ ]:
!pip install torch torchvision scikit-image tqdm matplotlib -q

from google.colab import drive
drive.mount('/content/drive')

import torch
print("CUDA:", torch.cuda.is_available())

Mounted at /content/drive
CUDA: False


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class AODnet(nn.Module):
    def __init__(self):
        super(AODnet, self).__init__()
        self.conv1 = nn.Conv2d(3,  3,  kernel_size=1, padding=0)
        self.conv2 = nn.Conv2d(3,  3,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(6,  3,  kernel_size=5, padding=2)
        self.conv4 = nn.Conv2d(6,  3,  kernel_size=7, padding=3)
        self.conv5 = nn.Conv2d(12, 3,  kernel_size=3, padding=1)
        self.b = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        x1 = F.relu(self.conv1(x))
        x2 = F.relu(self.conv2(x1))
        x3 = F.relu(self.conv3(torch.cat([x1, x2], dim=1)))
        x4 = F.relu(self.conv4(torch.cat([x2, x3], dim=1)))
        k  = F.relu(self.conv5(torch.cat([x1, x2, x3, x4], dim=1)))
        return k * x - k + self.b

In [ ]:
import os

os.makedirs("data/IGP_synthetic", exist_ok=True)
os.makedirs("data/SOTS_outdoor",  exist_ok=True)

print("Copying IGP dataset (~3-4 GB, takes 3-5 mins)...")
!cp -r "/content/drive/MyDrive/AODNet_data/IGP_synthetic/hazy"  data/IGP_synthetic/
!cp -r "/content/drive/MyDrive/AODNet_data/IGP_synthetic/clear" data/IGP_synthetic/

print("Copying SOTS test set...")
!cp -r "/content/drive/MyDrive/AODNet_data/SOTS/outdoor/hazy"  data/SOTS_outdoor/
!cp -r "/content/drive/MyDrive/AODNet_data/SOTS/outdoor/clear" data/SOTS_outdoor/

print("\nVerifying counts:")
print("IGP hazy :", len(os.listdir("data/IGP_synthetic/hazy")),  "images")
print("IGP clear:", len(os.listdir("data/IGP_synthetic/clear")), "images")
print("SOTS hazy:", len(os.listdir("data/SOTS_outdoor/hazy")),  "images")

Copying IGP dataset (~3-4 GB, takes 3-5 mins)...
cp: cannot stat '/content/drive/MyDrive/AODNet_data/IGP_synthetic/clear': No such file or directory
Copying SOTS test set...

Verifying counts:
IGP hazy : 14358 images


FileNotFoundError: [Errno 2] No such file or directory: 'data/IGP_synthetic/clear'

In [ ]:
import os
import shutil
from PIL import Image
import numpy as np

# ── Step 1: Check what hazy files we actually have ────────────────
hazy_files = os.listdir("data/IGP_synthetic/hazy")
print(f"Hazy images available: {len(hazy_files)}")
print(f"Sample: {hazy_files[:5]}")

Hazy images available: 14358
Sample: ['2076_light.png', '2288_dense.png', '4088_dense.png', '4587_dense.png', '1619_light.png']


In [ ]:
# Step 2 — but first copy RESIDE6K from Drive since it's not on Colab yet
import os
os.makedirs("data/RESIDE6K/train/GT", exist_ok=True)
print("Copying RESIDE6K GT from Drive...")
!cp -r "/content/drive/MyDrive/AODNet_data/RESIDE6K/RESIDE-6K/train/GT" \
    "data/RESIDE6K/train/"
print("GT images:", len(os.listdir("data/RESIDE6K/train/GT")))

Copying RESIDE6K GT from Drive...
GT images: 6000


In [ ]:
# ── Step 2: Rebuild clear folder from RESIDE-6K GT ───────────────
# Each hazy file is named like "1001_dense.png"
# Clear file is just "1001.jpg" from RESIDE-6K GT

os.makedirs("data/IGP_synthetic/clear", exist_ok=True)

gt_dir   = "data/RESIDE6K/train/GT"   # already on Colab from Drive
missing  = 0
rebuilt  = 0

for hazy_name in hazy_files:
    # Extract base number: "1001_dense.png" → "1001"
    stem      = hazy_name.rsplit('_', 1)[0]
    out_name  = hazy_name  # clear has same name as hazy for direct matching

    # Find source in GT folder
    clear_src = None
    for ext in ['.jpg', '.png', '.jpeg']:
        candidate = os.path.join(gt_dir, stem + ext)
        if os.path.exists(candidate):
            clear_src = candidate
            break

    if clear_src:
        # Resize to match 256x256 and save
        img = Image.open(clear_src).convert("RGB").resize((256, 256))
        img.save(f"data/IGP_synthetic/clear/{out_name}")
        rebuilt += 1
    else:
        missing += 1

print(f"Rebuilt : {rebuilt} clear images")
print(f"Missing : {missing} (GT source not found)")

Rebuilt : 14358 clear images
Missing : 0 (GT source not found)


In [ ]:
# ── Step 3: Save clear folder to Drive ───────────────────────────
print("Saving clear folder to Drive...")
!cp -r data/IGP_synthetic/clear \
    "/content/drive/MyDrive/AODNet_data/IGP_synthetic/clear"
print("Done")

Saving clear folder to Drive...
Done


In [ ]:
# ── Step 4: Verify final counts ──────────────────────────────────
hazy_count  = len(os.listdir("data/IGP_synthetic/hazy"))
clear_count = len(os.listdir("data/IGP_synthetic/clear"))
print(f"Hazy : {hazy_count}")
print(f"Clear: {clear_count}")
print(f"Match: {hazy_count == clear_count}")

Hazy : 14358
Clear: 14358
Match: True


In [ ]:
# SOTS test set
import os
os.makedirs("data/SOTS_outdoor", exist_ok=True)
!cp -r "/content/drive/MyDrive/AODNet_data/SOTS/outdoor/hazy"  data/SOTS_outdoor/
!cp -r "/content/drive/MyDrive/AODNet_data/SOTS/outdoor/clear" data/SOTS_outdoor/
print("SOTS hazy:", len(os.listdir("data/SOTS_outdoor/hazy")))
print("SOTS clear:", len(os.listdir("data/SOTS_outdoor/clear")))

SOTS hazy: 500
SOTS clear: 492


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
from tqdm import tqdm

# ── AODnet ────────────────────────────────────────────────────────
class AODnet(nn.Module):
    def __init__(self):
        super(AODnet, self).__init__()
        self.conv1 = nn.Conv2d(3,  3,  kernel_size=1, padding=0)
        self.conv2 = nn.Conv2d(3,  3,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(6,  3,  kernel_size=5, padding=2)
        self.conv4 = nn.Conv2d(6,  3,  kernel_size=7, padding=3)
        self.conv5 = nn.Conv2d(12, 3,  kernel_size=3, padding=1)
        self.b = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        x1 = F.relu(self.conv1(x))
        x2 = F.relu(self.conv2(x1))
        x3 = F.relu(self.conv3(torch.cat([x1, x2], dim=1)))
        x4 = F.relu(self.conv4(torch.cat([x2, x3], dim=1)))
        k  = F.relu(self.conv5(torch.cat([x1, x2, x3, x4], dim=1)))
        return k * x - k + self.b

# ── Dataset ───────────────────────────────────────────────────────
class DehazingDataset(Dataset):
    def __init__(self, hazy_dir, clear_dir, size=256, naming="direct"):
        self.hazy_dir  = hazy_dir
        self.clear_dir = clear_dir
        self.naming    = naming
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
        ])
        all_hazy = sorted([
            f for f in os.listdir(hazy_dir)
            if f.lower().endswith(('.jpg','.png','.jpeg'))
        ])
        self.pairs = []
        skipped = 0
        for hazy_name in all_hazy:
            clear_name = self._get_clear_name(hazy_name)
            if os.path.exists(os.path.join(clear_dir, clear_name)):
                self.pairs.append((hazy_name, clear_name))
            else:
                skipped += 1
        print(f"  {len(self.pairs)} valid pairs ({skipped} skipped)")

    def _get_clear_name(self, hazy_name):
        if self.naming == "reside":
            base = hazy_name.split('_')[0]
            for ext in ['.png', '.jpg', '.jpeg']:
                if os.path.exists(os.path.join(self.clear_dir, base + ext)):
                    return base + ext
            return base + '.png'
        return hazy_name

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        hazy_name, clear_name = self.pairs[idx]
        hazy  = Image.open(
            os.path.join(self.hazy_dir,  hazy_name)).convert('RGB')
        clear = Image.open(
            os.path.join(self.clear_dir, clear_name)).convert('RGB')
        return self.transform(hazy), self.transform(clear)

# ── Evaluate ──────────────────────────────────────────────────────
def evaluate(model, loader, device, n_batches=10):
    model.eval()
    psnr_scores, ssim_scores = [], []
    with torch.no_grad():
        for i, (hazy, clear) in enumerate(loader):
            if i >= n_batches: break
            output = model(hazy.to(device)).clamp(0, 1)
            for j in range(output.shape[0]):
                out_np = output[j].cpu().numpy().transpose(1,2,0)
                gt_np  = clear[j].numpy().transpose(1,2,0)
                psnr_scores.append(psnr_fn(gt_np, out_np, data_range=1.0))
                ssim_scores.append(ssim_fn(gt_np, out_np,
                                   channel_axis=2, data_range=1.0))
    return np.mean(psnr_scores), np.mean(ssim_scores)

# ── Fine-tune ─────────────────────────────────────────────────────
def finetune(model, train_loader, test_loader, epochs=10, lr=2e-4):
    device    = torch.device("cuda")
    model     = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='max', factor=0.5, patience=2)
    best_psnr = 0
    print(f"Fine-tuning on IGP data | {len(train_loader)} batches/epoch\n")

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for hazy, clear in tqdm(train_loader,
                                desc=f"Epoch {epoch+1:02d}/{epochs}"):
            hazy, clear = hazy.to(device), clear.to(device)
            optimizer.zero_grad()
            loss = criterion(model(hazy).clamp(0,1), clear)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        psnr_val, ssim_val = evaluate(model, test_loader, device)
        scheduler.step(psnr_val)

        print(f"  Epoch {epoch+1:02d} | Loss: {avg_loss:.6f} | "
              f"PSNR: {psnr_val:.2f} dB | SSIM: {ssim_val:.4f} | "
              f"b: {model.b.item():.4f}")

        if psnr_val > best_psnr:
            best_psnr = psnr_val
            torch.save(model.state_dict(),
                "/content/drive/MyDrive/AODNet_data/aodnet_igp_finetuned.pth")
            print(f"  ✓ Saved (PSNR: {best_psnr:.2f} dB)")

    print(f"\nDone. Best PSNR: {best_psnr:.2f} dB")
    return model

print("All definitions loaded successfully")

All definitions loaded successfully


In [ ]:
# Build loaders
print("IGP train:")
igp_dataset = DehazingDataset(
    hazy_dir = "data/IGP_synthetic/hazy",
    clear_dir = "data/IGP_synthetic/clear",
    naming   = "direct"
)
print("SOTS test:")
test_dataset = DehazingDataset(
    hazy_dir  = "data/SOTS_outdoor/hazy",
    clear_dir = "data/SOTS_outdoor/clear",
    naming    = "reside"
)

igp_loader  = DataLoader(igp_dataset, batch_size=64,
                         shuffle=True,  num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=8,
                         shuffle=False, num_workers=2, pin_memory=True)

IGP train:
  14358 valid pairs (0 skipped)
SOTS test:
  500 valid pairs (0 skipped)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
# Load baseline and confirm score
device = torch.device("cuda")
model  = AODnet().to(device)
model.load_state_dict(torch.load(
    "/content/drive/MyDrive/AODNet_data/aodnet_best.pth",
    map_location=device))
model.eval()

psnr_val, ssim_val = evaluate(model, test_loader, device)
print(f"Baseline — PSNR: {psnr_val:.2f} dB | SSIM: {ssim_val:.4f}")

Baseline — PSNR: 21.85 dB | SSIM: 0.9121


In [ ]:
# Fine-tune
model = finetune(model, igp_loader, test_loader, epochs=10, lr=2e-4)

Fine-tuning on IGP data | 225 batches/epoch



Epoch 01/10:   0%|          | 0/225 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epoch 01/10: 100%|██████████| 225/225 [03:05<00:00,  1.21it/s]


  Epoch 01 | Loss: 0.023688 | PSNR: 22.72 dB | SSIM: 0.9218 | b: 1.0614
  ✓ Saved (PSNR: 22.72 dB)


Epoch 02/10: 100%|██████████| 225/225 [03:00<00:00,  1.24it/s]


  Epoch 02 | Loss: 0.022050 | PSNR: 22.49 dB | SSIM: 0.9220 | b: 1.0765


...

  Epoch 08 | Loss: 0.020867 | PSNR: 21.54 dB | SSIM: 0.9173 | b: 1.1155


Epoch 09/10: 100%|██████████| 225/225 [02:44<00:00,  1.37it/s]


  Epoch 09 | Loss: 0.020817 | PSNR: 21.56 dB | SSIM: 0.9167 | b: 1.1171


Epoch 10/10: 100%|██████████| 225/225 [02:45<00:00,  1.36it/s]


  Epoch 10 | Loss: 0.020752 | PSNR: 21.63 dB | SSIM: 0.9172 | b: 1.1184

Done. Best PSNR: 22.72 dB


In [ ]:
import os

# Verify all critical files are on Drive
files_to_check = [
    "/content/drive/MyDrive/AODNet_data/aodnet_best.pth",
    "/content/drive/MyDrive/AODNet_data/aodnet_igp_finetuned.pth",
    "/content/drive/MyDrive/AODNet_data/IGP_synthetic/hazy",
    "/content/drive/MyDrive/AODNet_data/RESIDE6K/RESIDE-6K/train/GT",
    "/content/drive/MyDrive/AODNet_data/SOTS/outdoor/hazy",
]

print("Drive contents check:")
for path in files_to_check:
    exists = os.path.exists(path)
    print(f"  {'✓' if exists else '✗'} {path.split('AODNet_data/')[-1]}")

Drive contents check:
  ✓ aodnet_best.pth
  ✓ aodnet_igp_finetuned.pth
  ✓ IGP_synthetic/hazy
  ✓ RESIDE6K/RESIDE-6K/train/GT
  ✓ SOTS/outdoor/hazy


In [ ]:
!pip install torch torchvision scikit-image tqdm matplotlib -q

from google.colab import drive
drive.mount('/content/drive')

import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
CUDA: True
GPU: Tesla T4


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset
import torchvision.transforms as transforms
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
from tqdm import tqdm
import random

class AODnet(nn.Module):
    def __init__(self):
        super(AODnet, self).__init__()
        self.conv1 = nn.Conv2d(3,  3,  kernel_size=1, padding=0)
        self.conv2 = nn.Conv2d(3,  3,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(6,  3,  kernel_size=5, padding=2)
        self.conv4 = nn.Conv2d(6,  3,  kernel_size=7, padding=3)
        self.conv5 = nn.Conv2d(12, 3,  kernel_size=3, padding=1)
        self.b = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        x1 = F.relu(self.conv1(x))
        x2 = F.relu(self.conv2(x1))
        x3 = F.relu(self.conv3(torch.cat([x1, x2], dim=1)))
        x4 = F.relu(self.conv4(torch.cat([x2, x3], dim=1)))
        k  = F.relu(self.conv5(torch.cat([x1, x2, x3, x4], dim=1)))
        return k * x - k + self.b

class DehazingDataset(Dataset):
    def __init__(self, hazy_dir, clear_dir, size=256, naming="direct"):
        self.hazy_dir  = hazy_dir
        self.clear_dir = clear_dir
        self.naming    = naming
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
        ])
        all_hazy = sorted([
            f for f in os.listdir(hazy_dir)
            if f.lower().endswith(('.jpg','.png','.jpeg'))
        ])
        self.pairs = []
        skipped = 0
        for hazy_name in all_hazy:
            clear_name = self._get_clear_name(hazy_name)
            clear_path = os.path.join(clear_dir, clear_name)
            if not os.path.exists(clear_path):
                skipped += 1
                continue
            # Pre-validate image is readable
            try:
                with Image.open(clear_path) as img:
                    img.verify()
                self.pairs.append((hazy_name, clear_name))
            except Exception:
                skipped += 1
        print(f"  {len(self.pairs)} valid pairs ({skipped} skipped)")

    def _get_clear_name(self, hazy_name):
        if self.naming == "reside":
            base = hazy_name.split('_')[0]
            for ext in ['.png', '.jpg', '.jpeg']:
                if os.path.exists(os.path.join(self.clear_dir, base + ext)):
                    return base + ext
            return base + '.png'
        return hazy_name

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        hazy_name, clear_name = self.pairs[idx]
        try:
            hazy  = Image.open(
                os.path.join(self.hazy_dir,  hazy_name)).convert('RGB')
            clear = Image.open(
                os.path.join(self.clear_dir, clear_name)).convert('RGB')
            return self.transform(hazy), self.transform(clear)
        except Exception:
            # Return neighbour sample if this one fails
            return self.__getitem__((idx + 1) % len(self.pairs))

def evaluate(model, loader, device, n_batches=10):
    model.eval()
    psnr_scores, ssim_scores = [], []
    with torch.no_grad():
        for i, (hazy, clear) in enumerate(loader):
            if i >= n_batches: break
            output = model(hazy.to(device)).clamp(0, 1)
            for j in range(output.shape[0]):
                out_np = output[j].cpu().numpy().transpose(1,2,0)
                gt_np  = clear[j].numpy().transpose(1,2,0)
                psnr_scores.append(psnr_fn(gt_np, out_np, data_range=1.0))
                ssim_scores.append(ssim_fn(gt_np, out_np,
                                   channel_axis=2, data_range=1.0))
    return np.mean(psnr_scores), np.mean(ssim_scores)

def finetune(model, train_loader, test_loader, epochs=15, lr=5e-5):
    device    = torch.device("cuda")
    model     = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='max', factor=0.5, patience=3)
    best_psnr = 0
    print(f"Mixed fine-tuning | LR={lr} | {len(train_loader)} batches/epoch\n")

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for hazy, clear in tqdm(train_loader,
                                desc=f"Epoch {epoch+1:02d}/{epochs}"):
            hazy, clear = hazy.to(device), clear.to(device)
            optimizer.zero_grad()
            loss = criterion(model(hazy).clamp(0,1), clear)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        psnr_val, ssim_val = evaluate(model, test_loader, device)
        scheduler.step(psnr_val)

        print(f"  Epoch {epoch+1:02d} | Loss: {avg_loss:.6f} | "
              f"PSNR: {psnr_val:.2f} dB | SSIM: {ssim_val:.4f} | "
              f"b: {model.b.item():.4f}")

        if psnr_val > best_psnr:
            best_psnr = psnr_val
            torch.save(model.state_dict(),
                "/content/drive/MyDrive/AODNet_data/aodnet_mixed_finetuned.pth")
            print(f"  ✓ Saved (PSNR: {best_psnr:.2f} dB)")

    print(f"\nDone. Best PSNR: {best_psnr:.2f} dB")
    return model

print("All definitions loaded ✓")

All definitions loaded ✓


In [ ]:
os.makedirs("data/IGP_synthetic/hazy",  exist_ok=True)
os.makedirs("data/IGP_synthetic/clear", exist_ok=True)
os.makedirs("data/RESIDE6K/train/hazy", exist_ok=True)
os.makedirs("data/RESIDE6K/train/GT",   exist_ok=True)
os.makedirs("data/SOTS_outdoor",        exist_ok=True)

print("Copying IGP synthetic...")
!cp -r "/content/drive/MyDrive/AODNet_data/IGP_synthetic/hazy"  data/IGP_synthetic/
!cp -r "/content/drive/MyDrive/AODNet_data/IGP_synthetic/clear" data/IGP_synthetic/

print("Copying RESIDE-6K...")
!cp -r "/content/drive/MyDrive/AODNet_data/RESIDE6K/RESIDE-6K/train/hazy" data/RESIDE6K/train/
!cp -r "/content/drive/MyDrive/AODNet_data/RESIDE6K/RESIDE-6K/train/GT"   data/RESIDE6K/train/

print("Copying SOTS...")
!cp -r "/content/drive/MyDrive/AODNet_data/SOTS/outdoor/hazy"  data/SOTS_outdoor/
!cp -r "/content/drive/MyDrive/AODNet_data/SOTS/outdoor/clear" data/SOTS_outdoor/

print("\nCounts:")
print("IGP hazy  :", len(os.listdir("data/IGP_synthetic/hazy")))
print("IGP clear :", len(os.listdir("data/IGP_synthetic/clear")))
print("RESIDE hazy:", len(os.listdir("data/RESIDE6K/train/hazy")))
print("SOTS hazy  :", len(os.listdir("data/SOTS_outdoor/hazy")))

Copying IGP synthetic...
Copying RESIDE-6K...
Copying SOTS...

Counts:
IGP hazy  : 14358
IGP clear : 14358
RESIDE hazy: 6000
SOTS hazy  : 500


In [ ]:
print("SOTS test dataset:")
test_dataset = DehazingDataset(
    hazy_dir  = "data/SOTS_outdoor/hazy",
    clear_dir = "data/SOTS_outdoor/clear",
    naming    = "reside"
)
test_loader = DataLoader(test_dataset, batch_size=8,
                         shuffle=False, num_workers=2, pin_memory=True)

# Now evaluate on full test set
psnr_final, ssim_final = evaluate(model, test_loader,
                                   torch.device("cuda"), n_batches=62)
print(f"Final PSNR (full test set): {psnr_final:.2f} dB")
print(f"Final SSIM (full test set): {ssim_final:.4f}")

SOTS test dataset:
  499 valid pairs (1 skipped)
Final PSNR (full test set): 23.58 dB
Final SSIM (full test set): 0.9263


In [ ]:
device = torch.device("cuda")
model  = AODnet().to(device)
model.load_state_dict(torch.load(
    "/content/drive/MyDrive/AODNet_data/aodnet_best.pth",
    map_location=device))

psnr_val, ssim_val = evaluate(model, test_loader, device)
print(f"Baseline confirmed — PSNR: {psnr_val:.2f} dB | SSIM: {ssim_val:.4f}")
print("Starting mixed fine-tuning...\n")

model = finetune(model, mixed_loader, test_loader, epochs=15, lr=5e-5)

Baseline confirmed — PSNR: 21.85 dB | SSIM: 0.9121
Starting mixed fine-tuning...

Mixed fine-tuning | LR=5e-05 | 188 batches/epoch



Epoch 01/15: 100%|██████████| 188/188 [02:35<00:00,  1.21it/s]


  Epoch 01 | Loss: 0.024144 | PSNR: 23.55 dB | SSIM: 0.9240 | b: 1.0469
  ✓ Saved (PSNR: 23.55 dB)


Epoch 02/15: 100%|██████████| 188/188 [02:20<00:00,  1.33it/s]


  Epoch 02 | Loss: 0.023309 | PSNR: 23.60 dB | SSIM: 0.9254 | b: 1.0476
  ✓ Saved (PSNR: 23.60 dB)


...

  Epoch 13 | Loss: 0.023100 | PSNR: 23.63 dB | SSIM: 0.9276 | b: 1.0496


Epoch 14/15: 100%|██████████| 188/188 [02:04<00:00,  1.51it/s]


  Epoch 14 | Loss: 0.023077 | PSNR: 23.64 dB | SSIM: 0.9271 | b: 1.0497


Epoch 15/15: 100%|██████████| 188/188 [02:04<00:00,  1.51it/s]


  Epoch 15 | Loss: 0.023080 | PSNR: 23.65 dB | SSIM: 0.9274 | b: 1.0496

Done. Best PSNR: 23.67 dB


In [ ]:
# Verify the best model is on Drive
import os
path = "/content/drive/MyDrive/AODNet_data/aodnet_mixed_finetuned.pth"
size = os.path.getsize(path) / 1024
print(f"✓ Model saved: {size:.1f} KB")

# Quick final evaluation on full test set
psnr_final, ssim_final = evaluate(model, test_loader,
                                   torch.device("cuda"), n_batches=62)
print(f"Final PSNR (full test set): {psnr_final:.2f} dB")
print(f"Final SSIM (full test set): {ssim_final:.4f}")

✓ Model saved: 11.4 KB
Final PSNR (full test set): 23.58 dB
Final SSIM (full test set): 0.9263
